In [70]:
from google import genai
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import tqdm
import seaborn as sns

In [ ]:
train_df = pd.read_csv('Dataset/train_joined.csv')

In [44]:
train_df.head()

,Unnamed: 0,title,text,subject,date,label,joined
0,2619,excia head says trump remarks russia interfere...,former cia director john brennan friday critic...,politicsNews,"July 22, 2017",1,2619 excia head says trump remarks russia inte...
1,16043,’ believe punishment hispanic store owner swin...,man come store information much fraudster exce...,Government News,"Jun 19, 2017",0,16043 ’ believe punishment hispanic store owne...
2,876,federal reserve governor powells policy views ...,president donald trump thursday tapped federal...,politicsNews,"November 2, 2017",1,876 federal reserve governor powells policy vi...
3,19963,scoundrel hillary supporter starts “ trumpleak...,hillary clinton ally david brock offering pay ...,left-news,"Sep 17, 2016",0,19963 scoundrel hillary supporter starts “ tru...
4,10783,nancy pelosi arrogantly dismisses questions cr...,pleading ignorance perfect ploy nancy pelosi b...,politics,"May 26, 2017",0,10783 nancy pelosi arrogantly dismisses questi...


## pipeline

[x] Understand model limitations 2048 tokens --> [text](https://cloud.google.com/vertex-ai/generative-ai/docs/embeddings/get-text-embeddings?hl=it)  
 - This model has an attribute context that can accept strings or a list of strings.
 - 2048 tokens per string
 - 20000 tokens per request (so we can use 10 strings per each request)
 - Request rate: **5/min** and **100/day**

[x] Remove the articles that has more than 2048 tokens

[x] In this way we have 50 batches per day

### Understand model limitations

### Reducing the dataset with too many words

# OpenAI embeddings

In [1]:
import pandas
import tiktoken

In [16]:
df_train = pandas.read_csv('Dataset/train_cleaned.csv')
df_test = pandas.read_csv('Dataset/test_cleaned.csv')
df_validation = pandas.read_csv('Dataset/validation_cleaned.csv')

enc = tiktoken.encoding_for_model("text-embedding-ada-002")

df_total = pandas.concat([df_train, df_test, df_validation], ignore_index=True)

In [17]:
df_total['n_tokens'] = df_total['text'].apply(lambda x: len(enc.encode(x)))
total_tokens = df_total['n_tokens'].sum()

print(f"Token totali: {total_tokens}")

Token totali: 13575627


In [21]:
cost = total_tokens / 1000 * 0.0001
print(f"Total cost with OpenAI: {cost.round(2)} $")

Total cost with OpenAI: 1.36 $


In [33]:
from openai import OpenAI
import time
from tqdm import tqdm

In [ ]:


client = OpenAI(api_key=OPENAI_API_KEY)

## Formazione embeddings

In [29]:
def get_embedding(text, model="text-embedding-ada-002"):
    try:
        response = client.embeddings.create(
            input=[text],
            model=model
        )
        return response.data[0].embedding
    except Exception as e:
        print("Errore:", e)
        return None

In [41]:
# Train set embeddings
embeddings = []

for i, row in tqdm(df_train.iterrows(), total=len(df_train), desc="Generating embeddings"):
    emb = get_embedding(row['text'])
    embeddings.append(emb)

df_train['embedding'] = embeddings
df_train.to_csv("train_embeddings.csv", index=False)

Generating embeddings: 100%|██████████| 30000/30000 [3:21:48<00:00,  2.48it/s]   


In [40]:
# Test set embeddings
embeddings = []

for i, row in tqdm(df_test.iterrows(), total=len(df_test), desc="Generating embeddings"):
    emb = get_embedding(row['text'])
    embeddings.append(emb)

df_test['embedding'] = embeddings
df_test.to_csv("test_embeddings.csv", index=False)

Generating embeddings: 100%|██████████| 8267/8267 [1:14:50<00:00,  1.84it/s]   


In [ ]:
# Validation set embeddings

embeddings = []

for i, row in tqdm(df_validation.iterrows(), total=len(df_validation), desc="Generating embeddings"):
    emb = get_embedding(row['text'])
    embeddings.append(emb)

df_validation['embedding'] = embeddings
df_validation.to_csv("validation_embeddings.csv", index=False)


Generating embeddings: 100%|██████████| 6000/6000 [41:07<00:00,  2.43it/s]  
